# Concept Learning 

### Import and Load Dataset

In [5]:
import pandas as pd
from itertools import product

In [6]:
data = pd.read_csv("enjoysport.csv")

data

,sky,airtemp,humidity,wind,water,forcast,enjoysport
0,sunny,warm,normal,strong,warm,same,yes
1,sunny,warm,high,strong,warm,same,yes
2,rainy,cold,high,strong,warm,change,no
3,sunny,warm,high,strong,cool,change,yes


### Specifying Attribute, Feature, and Labels

In [7]:
attributes = [
    "sky",
    "airtemp",
    "humidity",
    "wind",
    "water",
    "forcast"
]

X = data[attributes].values
y = data["enjoysport"].values

print("Features:")
print(X)

print("\nLabels:")
print(y)

Features:
[['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
 ['sunny' 'warm' 'high' 'strong' 'warm' 'same']
 ['rainy' 'cold' 'high' 'strong' 'warm' 'change']
 ['sunny' 'warm' 'high' 'strong' 'cool' 'change']]

Labels:
<StringArray>
['yes', 'yes', 'no', 'yes']
Length: 4, dtype: str


## Find-S Algorithm

Implementing Find-S 

In [ ]:
def find_s(features, labels):

    hypothesis = None

    print("=== PROSES FIND-S ===")

    for i, (instance, label) in enumerate(
        zip(features, labels),
        start=1
    ):

        print(f"\nData ke-{i}")
        print("Instance:", instance)
        print("Label:", label)

        if label == "yes":

            if hypothesis is None:
                hypothesis = instance.copy()

            else:

                for j in range(len(hypothesis)):

                    if hypothesis[j] != instance[j]:
                        hypothesis[j] = "?"

        print("Hypothesis:", hypothesis)

    return hypothesis

Running Find-S

In [9]:
final_find_s = find_s(X, y)

print("\n=== FINAL HYPOTHESIS FIND-S ===")
print(final_find_s)

=== PROSES FIND-S ===

Data ke-1
Instance: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
Label: yes
Hypothesis: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']

Data ke-2
Instance: ['sunny' 'warm' 'high' 'strong' 'warm' 'same']
Label: yes
Hypothesis: ['sunny' 'warm' '?' 'strong' 'warm' 'same']

Data ke-3
Instance: ['rainy' 'cold' 'high' 'strong' 'warm' 'change']
Label: no
Hypothesis: ['sunny' 'warm' '?' 'strong' 'warm' 'same']

Data ke-4
Instance: ['sunny' 'warm' 'high' 'strong' 'cool' 'change']
Label: yes
Hypothesis: ['sunny' 'warm' '?' 'strong' '?' '?']

=== FINAL HYPOTHESIS FIND-S ===
['sunny' 'warm' '?' 'strong' '?' '?']


Domain Attribute

In [10]:
domains = []

for column in attributes:
    domains.append(list(data[column].unique()))

print("=== ATTRIBUTE DOMAINS ===")

for attribute, domain in zip(attributes, domains):
    print(f"{attribute}: {domain}")

=== ATTRIBUTE DOMAINS ===
sky: ['sunny', 'rainy']
airtemp: ['warm', 'cold']
humidity: ['normal', 'high']
wind: ['strong']
water: ['warm', 'cool']
forcast: ['same', 'change']


## Candidate Elimination

Helper Function

In [11]:
def covers(hypothesis, instance):

    for h, value in zip(hypothesis, instance):

        if h == "?":
            continue

        if h != value:
            return False

    return True


def hypothesis_more_general_or_equal(h1, h2):

    for a, b in zip(h1, h2):

        if a == "?":
            continue

        if a != b:
            return False

    return True


def generate_all_hypotheses(domains):

    possible_values = []

    for domain in domains:
        possible_values.append(domain + ["?"])

    return list(product(*possible_values))

Solving S (spesific) and G (general) Boundary

In [12]:
def get_specific_boundary(version_space):

    specific_boundary = []

    for h in version_space:

        is_most_specific = True

        for other in version_space:

            if h == other:
                continue

            if hypothesis_more_general_or_equal(h, other):
                is_most_specific = False
                break

        if is_most_specific:
            specific_boundary.append(h)

    return specific_boundary


def get_general_boundary(version_space):

    general_boundary = []

    for h in version_space:

        is_most_general = True

        for other in version_space:

            if h == other:
                continue

            if hypothesis_more_general_or_equal(other, h):
                is_most_general = False
                break

        if is_most_general:
            general_boundary.append(h)

    return general_boundary

Generating All Candidate Elimination Hypothesis

In [13]:
all_hypotheses = generate_all_hypotheses(domains)

print("Jumlah seluruh kemungkinan hypothesis:")
print(len(all_hypotheses))

Jumlah seluruh kemungkinan hypothesis:
486


Candidate Elimination with Step-by-Step Iteration

In [14]:
version_space = all_hypotheses.copy()


for i, (instance, label) in enumerate(
    zip(X, y),
    start=1
):

    print("\n" + "=" * 60)
    print(f"ITERASI KE-{i}")
    print("=" * 60)

    print("Instance:", instance)
    print("Label:", label)

    print(
        "\nJumlah hypothesis sebelum eliminasi:",
        len(version_space)
    )


    new_version_space = []


    for hypothesis in version_space:

        prediction = covers(hypothesis, instance)


        if label == "yes":

            if prediction:
                new_version_space.append(hypothesis)


        elif label == "no":

            if not prediction:
                new_version_space.append(hypothesis)


    version_space = new_version_space


    specific_boundary = get_specific_boundary(
        version_space
    )

    general_boundary = get_general_boundary(
        version_space
    )


    print(
        "\nJumlah hypothesis setelah eliminasi:",
        len(version_space)
    )


    print("\nSpecific Boundary (S):")

    for s in specific_boundary:
        print(s)


    print("\nGeneral Boundary (G):")

    for g in general_boundary:
        print(g)


ITERASI KE-1
Instance: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
Label: yes

Jumlah hypothesis sebelum eliminasi: 486

Jumlah hypothesis setelah eliminasi: 64

Specific Boundary (S):
('sunny', 'warm', 'normal', 'strong', 'warm', 'same')

General Boundary (G):
('?', '?', '?', '?', '?', '?')

ITERASI KE-2
Instance: ['sunny' 'warm' 'high' 'strong' 'warm' 'same']
Label: yes

Jumlah hypothesis sebelum eliminasi: 64

Jumlah hypothesis setelah eliminasi: 32

Specific Boundary (S):
('sunny', 'warm', '?', 'strong', 'warm', 'same')

General Boundary (G):
('?', '?', '?', '?', '?', '?')

ITERASI KE-3
Instance: ['rainy' 'cold' 'high' 'strong' 'warm' 'change']
Label: no

Jumlah hypothesis sebelum eliminasi: 32

Jumlah hypothesis setelah eliminasi: 28

Specific Boundary (S):
('sunny', 'warm', '?', 'strong', 'warm', 'same')

General Boundary (G):
('sunny', '?', '?', '?', '?', '?')
('?', 'warm', '?', '?', '?', '?')
('?', '?', '?', '?', '?', 'same')

ITERASI KE-4
Instance: ['sunny' 'warm' 'high'

Version Spaces

In [15]:
print("\nVersion Space:")

for h in version_space:
    print(h)


Version Space:
('sunny', 'warm', '?', 'strong', '?', '?')
('sunny', 'warm', '?', '?', '?', '?')
('sunny', '?', '?', 'strong', '?', '?')
('sunny', '?', '?', '?', '?', '?')
('?', 'warm', '?', 'strong', '?', '?')
('?', 'warm', '?', '?', '?', '?')


Final Result for Candidate Elimination Algorithm

In [16]:
final_S = get_specific_boundary(version_space)

final_G = get_general_boundary(version_space)


print("=== FINAL SPECIFIC BOUNDARY (S) ===")

for s in final_S:
    print(s)


print("\n=== FINAL GENERAL BOUNDARY (G) ===")

for g in final_G:
    print(g)


print("\n=== FINAL VERSION SPACE ===")

for h in version_space:
    print(h)


print("\nJumlah Final Version Space:")
print(len(version_space))

=== FINAL SPECIFIC BOUNDARY (S) ===
('sunny', 'warm', '?', 'strong', '?', '?')

=== FINAL GENERAL BOUNDARY (G) ===
('sunny', '?', '?', '?', '?', '?')
('?', 'warm', '?', '?', '?', '?')

=== FINAL VERSION SPACE ===
('sunny', 'warm', '?', 'strong', '?', '?')
('sunny', 'warm', '?', '?', '?', '?')
('sunny', '?', '?', 'strong', '?', '?')
('sunny', '?', '?', '?', '?', '?')
('?', 'warm', '?', 'strong', '?', '?')
('?', 'warm', '?', '?', '?', '?')

Jumlah Final Version Space:
6


Additional Testing Data

In [17]:
test_data = pd.DataFrame(
    [
        ["sunny", "warm", "normal", "strong", "cool", "same"],
        ["rainy", "warm", "high", "strong", "warm", "same"],
        ["sunny", "cold", "high", "strong", "warm", "change"],
        ["rainy", "cold", "normal", "strong", "cool", "same"]
    ],
    columns=attributes
)

test_data

,sky,airtemp,humidity,wind,water,forcast
0,sunny,warm,normal,strong,cool,same
1,rainy,warm,high,strong,warm,same
2,sunny,cold,high,strong,warm,change
3,rainy,cold,normal,strong,cool,same


Prediction Function

In [18]:
def predict_find_s(hypothesis, instance):

    if covers(hypothesis, instance):
        return "yes"

    return "no"


def predict_candidate_elimination(version_space, instance):

    predictions = []

    for hypothesis in version_space:
        predictions.append(
            covers(hypothesis, instance)
        )

    if all(predictions):
        return "yes"

    elif not any(predictions):
        return "no"

    else:
        return "unknown"

Running Prediction

In [19]:
find_s_predictions = []
candidate_predictions = []


for _, row in test_data.iterrows():

    instance = row.values


    find_s_result = predict_find_s(
        final_find_s,
        instance
    )

    candidate_result = (
        predict_candidate_elimination(
            version_space,
            instance
        )
    )


    find_s_predictions.append(
        find_s_result
    )

    candidate_predictions.append(
        candidate_result
    )


test_data["FIND-S Prediction"] = (
    find_s_predictions
)

test_data["Candidate Elimination Prediction"] = (
    candidate_predictions
)


test_data

,sky,airtemp,humidity,wind,water,forcast,FIND-S Prediction,Candidate Elimination Prediction
0,sunny,warm,normal,strong,cool,same,yes,yes
1,rainy,warm,high,strong,warm,same,no,unknown
2,sunny,cold,high,strong,warm,change,no,unknown
3,rainy,cold,normal,strong,cool,same,no,no


Berdasarkan hasil pengujian menggunakan beberapa data testing yang tidak terdapat pada dataset training, terlihat adanya perbedaan hasil prediksi antara algoritma FIND-S dan Candidate Elimination. Pada data testing pertama, kedua algoritma menghasilkan prediksi positif (`yes`). Hal ini menunjukkan bahwa data tersebut memenuhi hypothesis yang dihasilkan oleh FIND-S dan juga masih diterima oleh seluruh hypothesis yang terdapat dalam Version Space Candidate Elimination.

Pada data testing kedua dan ketiga, FIND-S menghasilkan prediksi negatif (`no`), sedangkan Candidate Elimination menghasilkan `unknown`. FIND-S memberikan hasil negatif karena kedua data tersebut tidak memenuhi salah satu atau lebih atribut spesifik pada final hypothesis yang dihasilkan. Karena FIND-S hanya memiliki satu final hypothesis, setiap data yang tidak tercakup oleh hypothesis tersebut secara langsung diklasifikasikan sebagai negatif.

Berbeda dengan FIND-S, Candidate Elimination mempertahankan beberapa hypothesis yang masih konsisten dengan seluruh data training. Pada data testing kedua dan ketiga, terdapat hypothesis dalam Version Space yang menerima data tersebut sebagai positif, sementara hypothesis lainnya menolaknya sebagai negatif. Perbedaan hasil antar hypothesis tersebut menyebabkan Candidate Elimination menghasilkan prediksi `unknown`. Hasil ini menunjukkan bahwa berdasarkan data training yang tersedia, belum terdapat informasi yang cukup untuk menentukan klasifikasi kedua data tersebut secara pasti.

Pada data testing keempat, kedua algoritma menghasilkan prediksi negatif (`no`). Hal ini menunjukkan bahwa data tersebut tidak memenuhi final hypothesis FIND-S dan juga tidak diterima oleh seluruh hypothesis yang masih berada dalam Version Space Candidate Elimination. Dengan demikian, kedua algoritma memberikan kesimpulan yang sama bahwa data tersebut lebih sesuai diklasifikasikan sebagai negatif berdasarkan pola yang diperoleh dari data training.

Dari hasil tersebut dapat disimpulkan bahwa FIND-S lebih sederhana dalam melakukan prediksi, tetapi keputusan yang dihasilkan hanya bergantung pada satu hypothesis dan tidak mempertimbangkan data negatif secara langsung. Candidate Elimination lebih kompleks karena menggunakan data positif dan negatif untuk membentuk batas Specific Boundary (S) dan General Boundary (G), tetapi mampu menunjukkan tingkat ketidakpastian ketika data training belum cukup untuk menghasilkan satu klasifikasi yang pasti.
